## Rainfall Data Exploration

### Crop Yield Risk & Advisory Dashboard

#### This notebook collects and explores district-level rainfall data for Uttar Pradesh, Punjab, and Haryana.

#### The goal is to create annual/seasonal rainfall indicators that can later be merged with historical crop production data.

In [1]:
import pandas as pd 


In [3]:
# Loading the dataset

file_path = "../../data/raw/rainfall/nasa_power_meerut_2021_2024.csv"

rainfall_raw = pd.read_csv(file_path , skiprows = 11)

In [4]:
rainfall_raw.head()

,YEAR,DOY,RH2M,QV2M,PRECTOTCORR
0,2021,1,30.98,2.62,0.00
1,2021,2,57.05,5.41,5.02
2,2021,3,87.84,9.84,15.60
3,2021,4,75.07,9.29,0.22
4,2021,5,74.14,9.29,6.13


### Basic profiling of the data 


In [5]:
rainfall_raw.shape

(1461, 5)

In [6]:
rainfall_raw.columns.tolist()

['YEAR', 'DOY', 'RH2M', 'QV2M', 'PRECTOTCORR']

In [8]:
rainfall_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1461 entries, 0 to 1460
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   YEAR         1461 non-null   int64  
 1   DOY          1461 non-null   int64  
 2   RH2M         1461 non-null   float64
 3   QV2M         1461 non-null   float64
 4   PRECTOTCORR  1461 non-null   float64
dtypes: float64(3), int64(2)
memory usage: 57.2 KB


In [9]:
rainfall_raw.describe()

,YEAR,DOY,RH2M,QV2M,PRECTOTCORR
count,1461.000000,1461.000000,1461.000000,1461.000000,1461.000000
mean,2022.501027,183.125257,55.084155,11.035250,2.680712
std,1.118723,105.474746,21.721134,6.238962,7.776861
min,2021.000000,1.000000,7.230000,1.330000,0.000000
25%,2022.000000,92.000000,37.780000,5.800000,0.000000
50%,2023.000000,183.000000,56.670000,8.710000,0.000000
75%,2024.000000,274.000000,73.770000,17.390000,0.960000
max,2024.000000,366.000000,93.800000,24.380000,93.950000


In [10]:
rainfall_raw.isnull().sum()

YEAR           0
DOY            0
RH2M           0
QV2M           0
PRECTOTCORR    0
dtype: int64

In [12]:
(rainfall_raw==-999).sum()

YEAR           0
DOY            0
RH2M           0
QV2M           0
PRECTOTCORR    0
dtype: int64

In [18]:
# Creating the actual date because dataset contain DOY

rainfall_raw["Date"] = (
    pd.to_datetime(
        rainfall_raw["YEAR"].astype(str),format="%Y")+
    pd.to_timedelta(rainfall_raw["DOY"]-1,unit="D"))

In [19]:
rainfall_raw.head()

,YEAR,DOY,RH2M,QV2M,PRECTOTCORR,Date
0,2021,1,30.98,2.62,0.00,2021-01-01
1,2021,2,57.05,5.41,5.02,2021-01-02
2,2021,3,87.84,9.84,15.60,2021-01-03
3,2021,4,75.07,9.29,0.22,2021-01-04
4,2021,5,74.14,9.29,6.13,2021-01-05


In [20]:
# Verifying the dates

rainfall_raw["Date"].min()

Timestamp('2021-01-01 00:00:00')

In [21]:
rainfall_raw["Date"].max()

Timestamp('2024-12-31 00:00:00')

In [22]:
# Creating the another dataset which doesn't contain moisture as not needed now 

rainfall_daily = rainfall_raw[
[
    "Date" , "YEAR" , "DOY", "PRECTOTCORR"]
].copy()

In [23]:
rainfall_daily.head()

,Date,YEAR,DOY,PRECTOTCORR
0,2021-01-01,2021,1,0.00
1,2021-01-02,2021,2,5.02
2,2021-01-03,2021,3,15.60
3,2021-01-04,2021,4,0.22
4,2021-01-05,2021,5,6.13


In [25]:
# Renaming the column prectotcorr to rainfall_mm

rainfall_daily = rainfall_daily.rename(
    columns = { 
        "PRECTOTCORR":"Rainfall_mm"}
)

In [26]:
rainfall_daily.head()

,Date,YEAR,DOY,Rainfall_mm
0,2021-01-01,2021,1,0.00
1,2021-01-02,2021,2,5.02
2,2021-01-03,2021,3,15.60
3,2021-01-04,2021,4,0.22
4,2021-01-05,2021,5,6.13


In [30]:
# Converting the daily rainfall data to annual rainfall 

annual_rainfall =  (
    rainfall_daily.groupby("YEAR" , as_index = False)["Rainfall_mm"].sum())

annual_rainfall

,YEAR,Rainfall_mm
0,2021,1124.21
1,2022,857.40
2,2023,1007.27
3,2024,927.64


In [31]:
# Reordering the dataset according to state/district

annual_rainfall["State"] = "Uttar_pradesh"
annual_rainfall["District"]="Meerut"



In [32]:
annual_rainfall = annual_rainfall[
[
    "State","District" , "YEAR" , "Rainfall_mm"]
]


In [33]:
annual_rainfall

,State,District,YEAR,Rainfall_mm
0,Uttar_pradesh,Meerut,2021,1124.21
1,Uttar_pradesh,Meerut,2022,857.40
2,Uttar_pradesh,Meerut,2023,1007.27
3,Uttar_pradesh,Meerut,2024,927.64


In [35]:
# Calculating the mean for rainfall

mean_rainfall = annual_rainfall["Rainfall_mm"].mean()

mean_rainfall

np.float64(979.13)

In [37]:
annual_rainfall["Rainfall_Deviation_From_4yr_Mean_Pct"] = (
    (
        annual_rainfall["Rainfall_mm"] - mean_rainfall
    )
    / mean_rainfall
) * 100

C:\Users\richa\AppData\Local\Temp\ipykernel_19120\843338206.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  annual_rainfall["Rainfall_Deviation_From_4yr_Mean_Pct"] = (


In [38]:
annual_rainfall

,State,District,YEAR,Rainfall_mm,Rainfall_Deviation_From_4yr_Mean_Pct
0,Uttar_pradesh,Meerut,2021,1124.21,14.817236
1,Uttar_pradesh,Meerut,2022,857.40,-12.432466
2,Uttar_pradesh,Meerut,2023,1007.27,2.873980
3,Uttar_pradesh,Meerut,2024,927.64,-5.258750
